# Demo - Client HTTP Resiliente para API de Pagamentos

> **⚠️ PRÉ-REQUISITO:** Inicie o mock server **antes** de executar as células:
> ```
> uv run uvicorn mock_server.server:app --port 8001
> ```
> Execute em um terminal separado e mantenha-o rodando durante toda a demonstração.

---

Este notebook demonstra todas as funcionalidades do `PaymentClient`:

| # | Seção | Operação |
|---|-------|----------|
| 1 | Setup | Imports, `nest_asyncio`, configurações ativas |
| 2 | Criar cobrança | `POST /charges` com sucesso |
| 3 | Consultar | `GET /charges/{id}` por ID |
| 4 | Listar | `GET /charges?page=1&per_page=5` paginado |
| 5 | Reembolsar | `POST /charges/{id}/refund` total |
| 6 | Retry (flaky) | Servidor instável → retry automático com backoff |
| 7 | Timeout | Servidor lento → `PaymentTimeoutError` capturada |
| 8 | Métricas | Latência média de N chamadas consecutivas |

Execute as células **em ordem**. O `charge_id` criado na seção 2 é usado nas seções 3 e 5.

## 1. Setup - Imports e Configuração

Aplica `nest_asyncio` para permitir `await` dentro do event loop já ativo do Jupyter.
Configura o `structlog` no formato **console** para logs legíveis nas células.

In [ ]:
import asyncio
import time
from decimal import Decimal

import nest_asyncio

# nest_asyncio permite chamar coroutines dentro do event loop do Jupyter,
# que já está em execução (sem isso, asyncio.run() lançaria RuntimeError).
nest_asyncio.apply()

from app.config import Settings
from app.client.payment_client import PaymentClient
from app.client.schemas import CreateChargeRequest, RefundRequest
from app.client.exceptions import (
    PaymentTimeoutError,
    PaymentUnavailableError,
    PaymentConnectionError,
    PaymentServiceError,
)

# Formato 'console' torna os logs do structlog legíveis com cores no Jupyter.
# Em produção usaríamos 'json' para ingestão em sistemas de observabilidade.
settings = Settings(log_format='console')

print('✓ Setup concluído!')
print(f'  URL base:         {settings.payment_api_base_url}')
print(f'  Connect timeout:  {settings.payment_api_connect_timeout}s')
print(f'  Read timeout:     {settings.payment_api_read_timeout}s')
print(f'  Max retries:      {settings.payment_api_max_retries}')
print(f'  Backoff base:     {settings.payment_api_backoff_base}s')

## 2. Criar Cobrança - `POST /charges`

Envia os dados via `CreateChargeRequest` (validado por Pydantic antes do HTTP).
O `PaymentClient` serializa `Decimal` como string para evitar imprecisão de ponto flutuante.

O `charge_id` retornado é armazenado na variável `charge_id` para uso nas seções seguintes.

In [ ]:
req = CreateChargeRequest(
    amount=Decimal('150.00'),
    currency='BRL',
    description='Pedido #1001 - Demonstração do client resiliente',
)

print(f'→ Enviando: amount=R${req.amount}, currency={req.currency}')
print()

async with PaymentClient(settings) as client:
    charge = await client.create_charge(req)

# Armazena o ID para uso nas seções de consulta e reembolso
charge_id = charge.id

print()
print('✓ Cobrança criada com sucesso!')
print(f'  ID:        {charge.id}')
print(f'  Valor:     R$ {charge.amount}')
print(f'  Status:    {charge.status}')
print(f'  Criada em: {charge.created_at}')

## 3. Consultar Cobrança - `GET /charges/{id}`

Busca o estado atual da cobrança pelo ID. O `PaymentClient` lança
`PaymentNotFoundError` (mapeada para HTTP 404) se o ID não existir.

In [ ]:
print(f'→ Consultando ID: {charge_id}')
print()

async with PaymentClient(settings) as client:
    fetched = await client.get_charge(charge_id)

print()
print('✓ Cobrança encontrada!')
print(f'  ID:        {fetched.id}')
print(f'  Valor:     R$ {fetched.amount}')
print(f'  Status:    {fetched.status}')
print(f'  Descrição: {fetched.description}')

## 4. Listar Cobranças - `GET /charges?page=1&per_page=5`

Retorna `ListChargesResponse` com os itens da página e metadados de paginação
(`total`, `page`, `per_page`). Permite navegar pelos resultados sem chamadas extras.

In [ ]:
async with PaymentClient(settings) as client:
    result = await client.list_charges(page=1, per_page=5)

total_pages = (
    (result.total + result.per_page - 1) // result.per_page
    if result.total > 0 else 1
)

print('✓ Listagem retornada!')
print(f'  Total de cobranças: {result.total}')
print(f'  Página atual:       {result.page} de {total_pages}')
print(f'  Itens nesta página: {len(result.items)}')
print()
for i, c in enumerate(result.items, 1):
    print(f'  [{i}] ID={c.id[:8]}... | R$ {c.amount} | status={c.status}')

## 5. Reembolsar Cobrança - `POST /charges/{id}/refund`

Processa o reembolso total da cobrança criada na seção 2.  
Sem informar `amount` no `RefundRequest`, a API usa o valor integral da cobrança.  
Após o reembolso, o status muda para `"refunded"` e novas tentativas retornam HTTP 400.

In [ ]:
req_refund = RefundRequest(charge_id=charge_id)

print(f'→ Solicitando reembolso total para ID: {charge_id}')
print()

async with PaymentClient(settings) as client:
    refund = await client.refund_charge(req_refund)

print()
print('✓ Reembolso processado!')
print(f'  ID do reembolso: {refund.id}')
print(f'  Cobrança:        {refund.charge_id}')
print(f'  Valor:           R$ {refund.amount}')
print(f'  Status:          {refund.status}')

## 6. Retry - Servidor Instável (Flaky)

O mock server mantém um contador global por sessão. Com `description='flaky'`:

- **1ª chamada** → HTTP 500 (`flaky attempt 1/2`)
- **2ª chamada** → HTTP 500 (`flaky attempt 2/2`)
- **3ª chamada** → HTTP 201 ✓ (sucesso -> contador resetado)

Observe nos logs abaixo os eventos `http_request_retriable_error` e
`http_request_retry_wait` mostrando o backoff exponencial entre as tentativas.

In [ ]:
req_flaky = CreateChargeRequest(
    amount=Decimal('99.90'),
    currency='BRL',
    description='flaky - teste de retry automático',
)

print("→ Enviando request com description='flaky'...")
print('  (Espere HTTP 500 nas 2 primeiras tentativas e 201 na 3ª)')
print()

t_start = time.perf_counter()
async with PaymentClient(settings) as client:
    charge_flaky = await client.create_charge(req_flaky)
elapsed = time.perf_counter() - t_start

print()
print(f'✓ Sucesso após retries! (tempo total: {elapsed:.2f}s)')
print(f'  ID:     {charge_flaky.id}')
print(f'  Status: {charge_flaky.status}')
print('  (O backoff exponencial adicionou delay entre as tentativas)')

## 7. Timeout - Servidor Sem Resposta

O mock server dorme **60 segundos** quando `description` contém `"timeout"`.  
Aqui usamos `read_timeout=3s` e `max_retries=2` para manter a demo rápida (~9s total).

O `PaymentClient` devolve `PaymentTimeoutError` após esgotar todos os retries.  
`status_code=None` porque não houve resposta HTTP, apenas expiração de timeout.

In [ ]:
# Settings específicos para esta demo: timeout curto + poucos retries.
# Em produção, read_timeout=30s e max_retries=3 são valores típicos.
timeout_settings = Settings(
    payment_api_read_timeout=3.0,
    payment_api_max_retries=2,
    payment_api_backoff_base=0.5,
    log_format='console',
)

req_timeout = CreateChargeRequest(
    amount=Decimal('50.00'),
    currency='BRL',
    description='timeout - demonstração de timeout',
)

print(f'→ Enviando request (timeout em {timeout_settings.payment_api_read_timeout}s por tentativa)...')
print(f'  Max retries: {timeout_settings.payment_api_max_retries}')
print()

t_start = time.perf_counter()
try:
    async with PaymentClient(timeout_settings) as client:
        await client.create_charge(req_timeout)
except PaymentTimeoutError as exc:
    elapsed = time.perf_counter() - t_start
    print()
    print(f'✓ PaymentTimeoutError capturada como esperado! ({elapsed:.1f}s total)')
    print(f'  Tipo:      {type(exc).__name__}')
    print(f'  Mensagem:  {exc.message}')
    print(f'  Retries:   {exc.retries_attempted}')
    print(f'  Status:    {exc.status_code}  ← None = sem resposta HTTP')

## 8. Métricas de Latência

Mede o tempo de resposta de `N` cobranças consecutivas, **reaproveitando o mesmo  
`PaymentClient`** (e portanto o mesmo `httpx.AsyncClient` com pool de conexões TCP).

Reutilizar o client evita o overhead de handshake TCP/TLS a cada chamada,
reduzindo a latência nas chamadas subsequentes.

In [ ]:
n_calls = 5
print(f'→ Executando {n_calls} cobranças e medindo latência de cada uma:')
print()

durations: list[float] = []

# Reusa o mesmo client para as N chamadas — aproveita o pool de conexões TCP
async with PaymentClient(settings) as client:
    for i in range(1, n_calls + 1):
        req_m = CreateChargeRequest(
            amount=Decimal(f'{i * 10}.00'),
            currency='BRL',
            description=f'Métrica #{i} de {n_calls}',
        )
        t_start = time.perf_counter()
        await client.create_charge(req_m)
        elapsed_ms = (time.perf_counter() - t_start) * 1000
        durations.append(elapsed_ms)
        print(f'  [{i}/{n_calls}] {elapsed_ms:.1f} ms')

avg = sum(durations) / len(durations)
min_d = min(durations)
max_d = max(durations)

print()
print(f'✓ Resultado das métricas ({n_calls} chamadas):')
print(f'  Média:   {avg:.1f} ms')
print(f'  Mínima:  {min_d:.1f} ms')
print(f'  Máxima:  {max_d:.1f} ms')
print(f'  Total:   {sum(durations):.1f} ms')